This module answers:

> Why can’t we just flatten images and use nn.Linear?

### Step 1. Build both models for images

Step 1. Build a NON-CNN model for images

In [18]:
import torch
from torchvision import datasets, transforms

transform = transforms.ToTensor()

train_data = datasets.MNIST(
    root="data",
    train=True,
    transform=transform,
    download=False
)

test_data = datasets.MNIST(
    root="data",
    train=False,
    download=False,
    transform=transform
)

from torch.utils.data import DataLoader

train_loader = DataLoader(train_data, batch_size=64, shuffle=True)
test_loader = DataLoader(test_data, batch_size=64, shuffle=False)

In [19]:
import torch.nn as nn

linear_model = nn.Sequential(
    nn.Flatten(),
    nn.Linear(28 * 28, 128),
    nn.ReLU(),
    nn.Linear(128, 10)
)

cnn_model = nn.Sequential(
    nn.Conv2d(1, 8, kernel_size=3),  # [1,28,28] → [8,26,26]
    nn.ReLU(),
    nn.MaxPool2d(2),    # [8,26,26] → [8,13,13]
    nn.Flatten(),    # [8,13,13] → [1352]
    nn.Linear(8 * 13 * 13, 10)  # 10 digit classes
)

### Step 2. Train and Evaluate

In [20]:
loss_fn = nn.CrossEntropyLoss()

def test_model(model):
    optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

    epochs = 3

    for epoch in range(epochs):
        model.train()
        
        for images, labels in train_loader:
            preds = model(images)
            loss = loss_fn(preds, labels)
            
            loss.backward()
            optimizer.step()
            optimizer.zero_grad()
            
        print(f"Epoch: {epoch} | Loss: {loss.item():.4f}")
        
    
    model.eval()
    correct = 0
    total = 0

    with torch.no_grad():
        for images, labels in test_loader:
            preds = model(images)
            predicted = preds.argmax(dim=1)
            
            correct += (predicted==labels).sum().item()
            total += labels.size(0)
            
    accuracy = correct / total
    print(accuracy)

### Step 3. Compare both models

In [22]:
print("Linear Model")
test_model(linear_model)

print("\n\nCNN")
test_model(cnn_model)

Linear Model
Epoch: 0 | Loss: 0.1334
Epoch: 1 | Loss: 0.0615
Epoch: 2 | Loss: 0.1785
0.9776


CNN
Epoch: 0 | Loss: 0.1093
Epoch: 1 | Loss: 0.1201
Epoch: 2 | Loss: 0.0845
0.9758


### Parameter comparison

Linear model:

`784 × 128 ≈ 100,000 parameters`


CNN:

`3×3×1×8 ≈ 72 parameters (first layer)`


CNN:
- learns more
- with less
- because structure matters